# GFP Brightness Classification from Protein Sequence

**Author:** Zahin Peerzade  
**Dataset:** GFP mutant library (UCEC course competition)  
**Task:** Binary classification: predict whether a mutated GFP sequence exhibits high brightness (class 1) or low brightness (class 0) from amino acid sequence alone  
**Primary Metric:** F1-score

---

## Overview

The inputs are protein sequences rather than tabular numeric features, so the central challenge is feature engineering: converting variable amino acid sequences into numeric representations that capture both positional and biochemical information.

The pipeline covers:

1. **Data loading and integrity checks** - shape verification, label distribution, sequence validation
2. **Feature engineering** - three complementary representations: one-hot encoding, amino acid composition frequencies, and physicochemical descriptor summaries
3. **Model comparison** - Logistic Regression, LinearSVC, Random Forest, HistGradientBoosting under 80/20 stratified split
4. **Regularization sweep** - LinearSVC C-parameter grid search
5. **Final model** - best LinearSVC retrained on all 31,029 training samples
6. **Prediction** - test set inference and submission generation

## 1. Imports

In [38]:
import os
os.chdir('/Users/zahinpeerzade/Downloads/gfp-brightness-classification')


import warnings
warnings.filterwarnings('ignore')

import os
import re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

## 2. Data Loading and Integrity Checks

The dataset contains:
- **31,029 training samples** with amino acid sequences and binary brightness labels
- **20,686 held-out test samples** (no labels)
- Labels: `0` = low brightness, `1` = high brightness

Key observations from exploration:
- All sequences are exactly **237 amino acids** long - no length variation
- All characters are the 20 canonical amino acids - no invalid characters
- Class distribution is moderately imbalanced: ~61% class 0, ~39% class 1

In [39]:
# Load data 
train_X = pd.read_csv('data/train_X.csv')
train_y = pd.read_csv('data/train_y.csv')
test_X  = pd.read_csv('data/test_X.csv')
sample_sub = pd.read_csv('data/y_sample_submission.csv')

print("train_X shape:", train_X.shape)
print("train_y shape:", train_y.shape)
print("test_X shape: ", test_X.shape)
print("\ntrain_X columns:", train_X.columns.tolist())
print("train_y columns:", train_y.columns.tolist())

train_X shape: (31029, 3)
train_y shape: (31029, 3)
test_X shape:  (20686, 2)

train_X columns: ['Unnamed: 0', 'ConstructedAASeq_cln', 'Id']
train_y columns: ['Unnamed: 0', 'Brightness_Class', 'Id']


In [40]:
# Drop redundant index columns, merge on ID
train_X_clean = train_X.drop(columns=['Unnamed: 0'], errors='ignore')
train_y_clean = train_y.drop(columns=['Unnamed: 0'], errors='ignore')
train_df = train_X_clean.merge(train_y_clean, on='Id')

print("Merged train_df shape:", train_df.shape)
print("\nMissing values:")
print(train_df.isnull().sum())

print("\nClass distribution:")
counts = train_df['Brightness_Class'].value_counts()
print(f"  Class 0 (low):  {counts[0]} ({counts[0]/len(train_df)*100:.1f}%)")
print(f"  Class 1 (high): {counts[1]} ({counts[1]/len(train_df)*100:.1f}%)")

Merged train_df shape: (31029, 3)

Missing values:
ConstructedAASeq_cln    0
Id                      0
Brightness_Class        0
dtype: int64

Class distribution:
  Class 0 (low):  18948 (61.1%)
  Class 1 (high): 12081 (38.9%)


In [41]:
# Verify sequence lengths and character validity
train_df['seq_len'] = train_df['ConstructedAASeq_cln'].apply(len)
test_X['seq_len']   = test_X['ConstructedAASeq_cln'].apply(len)

print("Training sequence lengths:", train_df['seq_len'].unique())
print("Test sequence lengths:    ", test_X['seq_len'].unique())

def valid_aa(seq):
    return bool(re.fullmatch(r'[ACDEFGHIKLMNPQRSTVWY]+', seq))

print(f"\nInvalid sequences - train: {(~train_df['ConstructedAASeq_cln'].apply(valid_aa)).sum()}")
print(f"Invalid sequences - test:  {(~test_X['ConstructedAASeq_cln'].apply(valid_aa)).sum()}")

Training sequence lengths: [237]
Test sequence lengths:     [237]

Invalid sequences - train: 0
Invalid sequences - test:  0


## 3. Feature Engineering

Three complementary feature sets are constructed and concatenated into a single sparse feature matrix.

### 3a. One-Hot Encoding (positional identity)

Each of the 237 positions is one-hot encoded independently. This lets the model learn the effect of specific residues at specific positions -- the most direct representation of positional mutation effects. With 237 positions and up to ~20 observed amino acids per position, this produces **1,930 sparse features**.

In [42]:
# Split sequences into character arrays
train_seqs = train_df['ConstructedAASeq_cln'].apply(list).tolist()
test_seqs  = test_X['ConstructedAASeq_cln'].apply(list).tolist()

train_df_seq = pd.DataFrame(train_seqs)
test_df_seq  = pd.DataFrame(test_seqs)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
ohe_train = ohe.fit_transform(train_df_seq)
ohe_test  = ohe.transform(test_df_seq)

print("OHE train shape:", ohe_train.shape)
print("OHE test shape: ", ohe_test.shape)

OHE train shape: (31029, 1930)
OHE test shape:  (20686, 1930)


### 3b. Amino Acid Composition Frequencies (global composition)

For each sequence, compute the frequency of each of the 20 canonical amino acids across all 237 positions. This captures the overall biochemical character of the mutant (e.g., how hydrophobic, charged, or small the overall protein is) without position-specific information. Produces **20 features** per sequence.

In [43]:
AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

def aa_freq_features(seq):
    L = len(seq)
    return pd.Series({aa: seq.count(aa) / L for aa in AMINO_ACIDS})

aa_train = train_df['ConstructedAASeq_cln'].apply(aa_freq_features)
aa_test  = test_X['ConstructedAASeq_cln'].apply(aa_freq_features)

print("AA composition train shape:", aa_train.shape)
print("AA composition test shape: ", aa_test.shape)

AA composition train shape: (31029, 20)
AA composition test shape:  (20686, 20)


### 3c. Physicochemical Descriptor Summaries

Six established amino acid descriptor tables are loaded from the `data/descriptors/` folder. Each table encodes biochemical properties per amino acid:

| Descriptor | Source | Properties encoded |
|---|---|---|
| DPPS | Tian et al. 2008 | Electronic, steric, hydrophobic, H-bond (10 PCs) |
| MS-WHIM | Zaliani & Gancia 1999 | 3D surface-derived steric and electrostatic (3 PCs) |
| Physical | Barley et al. 2018 | Hydrophilicity and volume (2 scales) |
| ST-scale | Yang et al. 2010 | Structural topology (8 PCs from 827 variables) |
| VHSE-scale | Mei et al. 2005 | Hydrophobic, steric, electronic (8 PCs) |
| Z-scale | Hellberg et al. 1987 | Hydrophilicity, bulk, electronic (3 PCs) |

For each descriptor table and each descriptor column, four summary statistics are computed across the 237-residue sequence: **mean, standard deviation, min, max**. This produces **136 features** per sequence capturing aggregate physicochemical properties.

In [44]:
descriptor_folder = 'data/descriptors'
descriptor_files = ['DPPS.csv', 'MS-WHIM.csv', 'Physical.csv', 'ST-scale.csv', 'Z-scale.csv', 'VHSE-scale.csv']

descriptor_tables = {}
for fname in descriptor_files:
    path = os.path.join(descriptor_folder, fname)
    df = pd.read_csv(path, header=None)
    df = df[df[1].isin(AMINO_ACIDS)]
    df = df.set_index(1)
    df = df.drop(columns=[0], errors='ignore')
    df = df.apply(lambda col: pd.to_numeric(col, errors='coerce'))
    df = df.dropna(axis=1, how='all')  # drop columns that are entirely NaN
    df = df.fillna(df.mean())
    df.columns = [f'f{i}' for i in range(1, len(df.columns) + 1)]
    df = df.loc[AMINO_ACIDS]
    descriptor_tables[fname] = df
    
   
print("Descriptor tables loaded:")
for name, df in descriptor_tables.items():
    print(f"  {name}: {df.shape}")

Descriptor tables loaded:
  DPPS.csv: (20, 10)
  MS-WHIM.csv: (20, 3)
  Physical.csv: (20, 2)
  ST-scale.csv: (20, 8)
  Z-scale.csv: (20, 3)
  VHSE-scale.csv: (20, 8)


In [45]:
def descriptor_features_for_sequence(seq, descriptor_dict):
    """Compute mean/std/min/max of each descriptor column across all residues in seq."""
    features = {}
    aa_list = list(seq)
    for name, df in descriptor_dict.items():
        mat = df.loc[aa_list].values
        for col_idx in range(mat.shape[1]):
            col_vals = mat[:, col_idx].astype(float)
            features[f'{name}_f{col_idx+1}_mean'] = col_vals.mean()
            features[f'{name}_f{col_idx+1}_std']  = col_vals.std()
            features[f'{name}_f{col_idx+1}_min']  = col_vals.min()
            features[f'{name}_f{col_idx+1}_max']  = col_vals.max()
    return pd.Series(features)

print("Computing descriptor features for training set...")
descriptor_train = train_df['ConstructedAASeq_cln'].apply(
    lambda seq: descriptor_features_for_sequence(seq, descriptor_tables)
)
print("Computing descriptor features for test set...")
descriptor_test = test_X['ConstructedAASeq_cln'].apply(
    lambda seq: descriptor_features_for_sequence(seq, descriptor_tables)
)

print(f"\nDescriptor train shape: {descriptor_train.shape}")
print(f"Descriptor test shape:  {descriptor_test.shape}")

Computing descriptor features for training set...
Computing descriptor features for test set...

Descriptor train shape: (31029, 136)
Descriptor test shape:  (20686, 136)


### 3d. Concatenate All Features

The three feature sets are concatenated horizontally into a single sparse matrix:
- OHE features: 1,930
- AA composition: 20
- Physicochemical descriptors: 136
- **Total: 2,086 features per sequence**

In [46]:
aa_train_sparse          = csr_matrix(aa_train.values)
aa_test_sparse           = csr_matrix(aa_test.values)
descriptor_train_sparse  = csr_matrix(descriptor_train.values)
descriptor_test_sparse   = csr_matrix(descriptor_test.values)

X_train = hstack([ohe_train, aa_train_sparse, descriptor_train_sparse])
X_test  = hstack([ohe_test,  aa_test_sparse,  descriptor_test_sparse])
y_train = train_df['Brightness_Class'].values
test_ids = test_X['Id'].values

print("FINAL X_train shape:", X_train.shape)
print("FINAL X_test shape: ", X_test.shape)
print("y_train shape:      ", y_train.shape)

FINAL X_train shape: (31029, 2086)
FINAL X_test shape:  (20686, 2086)
y_train shape:       (31029,)


## 4. Model Comparison

Four model families are compared on an 80/20 stratified train/validation split. Stratification preserves the ~61/39 class ratio across both splits.

The primary metric is **F1-score** because the classes are imbalanced and precision/recall balance matters more than raw accuracy.

In [47]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)
print("Train split:", X_tr.shape, y_tr.shape)
print("Val split:  ", X_val.shape, y_val.shape)

Train split: (24823, 2086) (24823,)
Val split:   (6206, 2086) (6206,)


In [48]:
# Logistic Regression
print("Training Logistic Regression (saga solver, L2):")
logreg = LogisticRegression(max_iter=5000, solver='saga', penalty='l2', n_jobs=-1)
logreg.fit(X_tr, y_tr)
y_pred_log = logreg.predict(X_val)
print(f"  F1:       {f1_score(y_val, y_pred_log):.4f}")
print(f"  Accuracy: {accuracy_score(y_val, y_pred_log):.4f}")
print(f"  Confusion matrix:\n{confusion_matrix(y_val, y_pred_log)}")

Training Logistic Regression (saga solver, L2):
  F1:       0.8625
  Accuracy: 0.8882
  Confusion matrix:
[[3335  455]
 [ 239 2177]]


In [49]:
# LinearSVC
print("Training LinearSVC (C=1.0):")
svm_model = LinearSVC(C=1.0, max_iter=5000)
svm_model.fit(X_tr, y_tr)
y_pred_svm = svm_model.predict(X_val)
print(f"  F1:       {f1_score(y_val, y_pred_svm):.4f}")
print(f"  Accuracy: {accuracy_score(y_val, y_pred_svm):.4f}")
print(f"  Confusion matrix:\n{confusion_matrix(y_val, y_pred_svm)}")

Training LinearSVC (C=1.0):
  F1:       0.8653
  Accuracy: 0.8912
  Confusion matrix:
[[3363  427]
 [ 248 2168]]


In [50]:
# Random Forest
print("Training Random Forest (300 trees, max_depth=20):")
rf_model = RandomForestClassifier(n_estimators=300, max_depth=20, n_jobs=-1, random_state=42)
rf_model.fit(X_tr.toarray(), y_tr)
y_pred_rf = rf_model.predict(X_val.toarray())
print(f"  F1:       {f1_score(y_val, y_pred_rf):.4f}")
print(f"  Accuracy: {accuracy_score(y_val, y_pred_rf):.4f}")
print(f"  Confusion matrix:\n{confusion_matrix(y_val, y_pred_rf)}")


Training Random Forest (300 trees, max_depth=20):
  F1:       0.6909
  Accuracy: 0.7606
  Confusion matrix:
[[3059  731]
 [ 755 1661]]


In [51]:
# HistGradientBoosting
print("Training HistGradientBoostingClassifier:")
X_tr_dense  = X_tr.toarray().astype(np.float32)
X_val_dense = X_val.toarray().astype(np.float32)

hgb = HistGradientBoostingClassifier(
    learning_rate=0.1, max_depth=10, max_iter=300,
    random_state=42, l2_regularization=0.1, early_stopping=True
)
hgb.fit(X_tr_dense, y_tr)
y_pred_hgb = hgb.predict(X_val_dense)
print(f"  F1:       {f1_score(y_val, y_pred_hgb):.4f}")
print(f"  Accuracy: {accuracy_score(y_val, y_pred_hgb):.4f}")
print(f"  Confusion matrix:\n{confusion_matrix(y_val, y_pred_hgb)}")

Training HistGradientBoostingClassifier:
  F1:       0.8276
  Accuracy: 0.8605
  Confusion matrix:
[[3261  529]
 [ 337 2079]]


**Summary:** LinearSVC and Logistic Regression outperform tree-based models (Random Forest F1 ~0.69, HistGradientBoosting ~0.83) on this high-dimensional sparse feature space. This is expected -- linear classifiers handle sparse, high-dimensional inputs more gracefully than ensemble methods that rely on feature splits.

## 5. Regularization Sweep -- LinearSVC C Parameter

The regularization strength `C` controls the trade-off between margin width and training error. Smaller `C` = stronger regularization (bigger margin, more misclassifications tolerated). We sweep 5 values to find the optimal setting.

In [52]:
Cs = [0.1, 0.5, 1.0, 2.0, 5.0]
results = {}

print("LinearSVC C sweep:")
for C in Cs:
    model = LinearSVC(C=C, max_iter=6000)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)
    f1 = f1_score(y_val, preds)
    results[C] = f1
    print(f"  C={C}:  F1={f1:.4f}")

best_C = max(results, key=results.get)
print(f"\nBest C: {best_C}  (F1={results[best_C]:.4f})")

LinearSVC C sweep:
  C=0.1:  F1=0.8604
  C=0.5:  F1=0.8660
  C=1.0:  F1=0.8653
  C=2.0:  F1=0.8646
  C=5.0:  F1=0.8633

Best C: 0.5  (F1=0.8660)


## 6. Final Model -- Retrain on Full Training Set

The best LinearSVC configuration is retrained on all 31,029 training samples. This maximizes the data available for learning before predicting on the 20,686 held-out test samples.

In [53]:
print(f"Training final LinearSVC (C={best_C}) on ALL training data...")
final_model = LinearSVC(C=best_C, max_iter=6000)
final_model.fit(X_train, y_train)
print("Final model trained.")

Training final LinearSVC (C=0.5) on ALL training data...
Final model trained.


## 7. Test Set Prediction and Submission

In [54]:
test_pred = final_model.predict(X_test)

print("Test set predictions:")
print(f"  Total:             {len(test_pred)}")
print(f"  Predicted class 1: {(test_pred == 1).sum()} ({(test_pred == 1).mean()*100:.1f}%)")
print(f"  Predicted class 0: {(test_pred == 0).sum()} ({(test_pred == 0).mean()*100:.1f}%)")

submission = pd.DataFrame({
    'Id': test_ids,
    'Brightness_Class': test_pred
})
submission.to_csv('results/final.csv', index=False)
print("\nSaved to results/final.csv")
print(submission.head())

Test set predictions:
  Total:             20686
  Predicted class 1: 8657 (41.8%)
  Predicted class 0: 12029 (58.2%)

Saved to results/final.csv
      Id  Brightness_Class
0  50579                 1
1  37987                 1
2  53977                 0
3  10677                 0
4  35653                 1
